# Arbitrage with Put-Call Parity

## Data

Use the data in `data/option_data_bb_NVDA.xlsx`
* tab `spot`: the underlying NVDA quote
* one tab per expiration date, (e.g. `2026-09-18`,): the option chain for that expiration

In [1]:
import sys
sys.path.insert(0, '../cmds')

import numpy as np
import pandas as pd
from pandas import IndexSlice
import matplotlib.pyplot as plt

from options import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 14

In [2]:
TICK = 'NVDA'
EXPRY = '2026-09-18'

FILEDATA = '../data/option_data_bb_NVDA.xlsx'

spot = pd.read_excel(FILEDATA, sheet_name='spot')
spot.rename(columns={'Unnamed: 0': 'field'}, inplace=True)
spot.set_index('field', inplace=True)

opt = pd.read_excel(FILEDATA, sheet_name=EXPRY)
opt.rename(columns={'Unnamed: 0': 'ticker'}, inplace=True)
opt.set_index('ticker', inplace=True)

In [3]:
styled = (
    spot.style
    .format("{:.2f}", subset=IndexSlice[['hist vol 30d', 'hist vol 60d', 'price'], :])
    .format("{:%Y-%m-%d}", subset=IndexSlice[['last update date'], :])
    .format("{:,.0f}", subset=IndexSlice[['volume'], :])
)

display(styled)

,NVDA US Equity
field,
name,NVIDIA Corp
last update date,2026-06-08
last update time,20:56:12.393006
price,208.64
volume,"138,372,837"
hist vol 30d,46.01
hist vol 60d,40.87


In [4]:
styled = (
    opt.iloc[1::8].style
    .format("{:.2f}", subset=IndexSlice[:, ['strike price', 'price', 'finance rate', 'implied vol', 'delta', 'gamma', 'vega', 'theta', 'rho', 'bid', 'ask', 'bid size', 'ask size']])
    .format(lambda x: x.strftime("%Y-%m-%d") if pd.notna(x) else "", subset=IndexSlice[:, ['last update date']], na_rep='')
    .format("{:,.0f}", subset=IndexSlice[:, ['volume', 'open int']])
)
display(styled)

,last update date,last update time,days to expiration,option type,strike price,price,finance rate,implied vol,delta,gamma,vega,theta,rho,bid,ask,bid size,ask size,open int,volume
ticker,,,,,,,,,,,,,,,,,,,
NVDA US 09/18/26 C135 Equity,2026-06-08,20:56:12.393006,102,Call,135.00,76.52,0.04,0.55,0.95,0.00,0.11,-0.02,0.00,75.55,77.15,66.00,42.00,"2,416",1
NVDA US 09/18/26 C175 Equity,2026-06-08,20:56:12.393006,102,Call,175.00,40.87,0.04,0.47,0.81,0.01,0.30,-0.06,0.00,41.35,41.95,33.00,40.00,"8,804",222
NVDA US 09/18/26 C215 Equity,2026-06-08,20:56:12.393006,102,Call,215.00,17.21,0.04,0.44,0.51,0.02,0.44,-0.09,0.00,17.40,17.60,62.00,22.00,"9,183",429
NVDA US 09/18/26 C255 Equity,2026-06-08,20:56:12.393006,102,Call,255.00,6.05,0.04,0.44,0.24,0.01,0.34,-0.07,0.00,6.10,6.25,102.00,42.00,"36,550",337
NVDA US 09/18/26 P140 Equity,2026-06-08,20:56:12.393006,102,Put,140.00,1.49,0.04,0.53,-0.05,0.00,0.12,-0.03,-0.00,1.47,1.51,185.00,155.00,"29,358",230
NVDA US 09/18/26 P180 Equity,2026-06-08,20:56:12.393006,102,Put,180.00,7.10,0.04,0.46,-0.22,0.01,0.33,-0.07,-0.00,7.10,7.30,112.00,73.00,"29,639","3,302"
NVDA US 09/18/26 P220 Equity,2026-06-08,20:56:12.393006,102,Put,220.00,25.05,0.04,0.43,-0.54,0.02,0.44,-0.09,-0.00,24.35,24.60,56.00,60.00,"8,335","1,097"
NVDA US 09/18/26 P260 Equity,2026-06-05,20:56:12.393006,102,Put,260.00,58.03,0.04,0.43,-0.81,0.01,0.30,-0.07,-0.00,54.10,55.35,33.00,14.00,221,19


#### A note on dividends

NVDA pays a small quarterly dividend (recently raised). Bloomberg's `finance rate` is an *implied* financing rate backed out from the option mids; for a dividend payer it reflects the **net** cost of carry (financing minus the dividend), so the put-call-parity check below already accounts for NVDA's dividend without a separate term. (Contrast this with the Black-Scholes exercise, whose AMZN options are on a stock that pays **no** dividend.)

# 1. Put-Call-Parity

Consider the option chain with expiration `2026-09-18`.

### 1.1.

For every listed strike, calculate how closely put-call parity holds. That is, compare the call spread $c-p$ to the present value moneyness, $S-K^*$.

Make a chart showing this error across strikes.

#### Note: Financing Rate

The financing rate is quoted from annual swaps with ACT/360 compounding. Thus, a quoted rate of $r$ and days-to-expiration of $\text{days}$ would be used to calculate the present value of some amount, $X$, as...

$$PV(X) = \frac{X}{1+\frac{\text{days}}{360} r}$$

**Answer 1.1.** Put-call parity is
\[
C-P=S-PV(K), \qquad
PV(K)=\frac{K}{1+\frac{\text{days}}{360}r}.
\]
The cell below computes the error
\[
(C-P)-(S-PV(K))
\]
at every strike. A positive error means the call-minus-put spread is rich relative to parity; a negative error means it is cheap.


In [ ]:
S = float(spot.loc['price'].iloc[0])

calls = (
    opt[opt['option type'].astype(str).str.lower().eq('call')]
    .copy()
    .set_index('strike price')
    .sort_index()
)
puts = (
    opt[opt['option type'].astype(str).str.lower().eq('put')]
    .copy()
    .set_index('strike price')
    .sort_index()
)

strikes = calls.index.intersection(puts.index).sort_values()

parity = pd.DataFrame(index=strikes)
parity.index.name = 'strike'
parity['call price'] = calls.loc[strikes, 'price'].astype(float)
parity['put price'] = puts.loc[strikes, 'price'].astype(float)
parity['days'] = (
    calls.loc[strikes, 'days to expiration'].astype(float).values
    + puts.loc[strikes, 'days to expiration'].astype(float).values
) / 2
parity['finance rate'] = (
    calls.loc[strikes, 'finance rate'].astype(float).values
    + puts.loc[strikes, 'finance rate'].astype(float).values
) / 2

parity['PV(K)'] = parity.index.to_numpy(dtype=float) / (
    1 + parity['days'] / 360 * parity['finance rate']
)
parity['call - put'] = parity['call price'] - parity['put price']
parity['S - PV(K)'] = S - parity['PV(K)']
parity['parity error'] = parity['call - put'] - parity['S - PV(K)']

display(
    parity[['call price', 'put price', 'PV(K)', 'call - put',
            'S - PV(K)', 'parity error']]
    .style.format('{:.4f}')
)

plt.figure()
plt.plot(parity.index, parity['parity error'], marker='o')
plt.axhline(0, linewidth=1)
plt.xlabel('Strike')
plt.ylabel('Put-call parity error ($/share)')
plt.title(f'{TICK} {EXPRY}: Put-Call Parity Error')
plt.grid(alpha=0.25)
plt.show()


### 1.2. Put-Call Arbitrage

Suppose you are going to put on a trade 
* sized to `1,000` long-short options contracts. 
* trading on the `ATM strike`, which is `210`.

(Recall that an option contract size is `100` shares, so the position will be `100,000` shares.)

Describe in detail the position you would take, including your positioning in the...
* calls
* puts
* shares
* cash

**Answer 1.2.** For an executable arbitrage check, long options are bought at the **ask** and short options are sold at the **bid**. The stock leg uses the quoted spot price because the supplied `spot` sheet gives one stock price rather than a bid/ask.

There are two candidate packages:

- **Conversion:** long stock + long put − short call. It pays exactly \(K\) per share at expiration.
- **Reversal:** short stock + long call − short put. It pays exactly \(-K\) per share at expiration.

The code compares both and prints the better trade at \(K=210\) for 1,000 contracts = 100,000 shares.


In [ ]:
N_CONTRACTS = 1_000
CONTRACT_SIZE = 100
N_SHARES = N_CONTRACTS * CONTRACT_SIZE
ATM = 210.0

def option_row(K, kind):
    mask = (
        np.isclose(opt['strike price'].astype(float), float(K))
        & opt['option type'].astype(str).str.lower().eq(kind.lower())
    )
    rows = opt.loc[mask]
    if len(rows) != 1:
        raise ValueError(f'Expected one {kind} at strike {K}; found {len(rows)}.')
    return rows.iloc[0]

def arb_stats(K):
    c = option_row(K, 'call')
    p = option_row(K, 'put')

    days = float(np.mean([c['days to expiration'], p['days to expiration']]))
    r = float(np.mean([c['finance rate'], p['finance rate']]))
    growth = 1 + days / 360 * r
    pv_k = float(K) / growth

    # Conversion: buy stock, buy put at ask, sell call at bid.
    conv_cost = S + float(p['ask']) - float(c['bid'])
    conv_pv_edge = pv_k - conv_cost
    conv_terminal_profit = (float(K) - conv_cost * growth) * N_SHARES

    # Reversal: short stock, buy call at ask, sell put at bid.
    rev_proceeds = S - float(c['ask']) + float(p['bid'])
    rev_pv_edge = rev_proceeds - pv_k
    rev_terminal_profit = (rev_proceeds * growth - float(K)) * N_SHARES

    if conv_terminal_profit >= rev_terminal_profit:
        best = 'conversion'
        best_terminal_profit = conv_terminal_profit
        best_pv_edge = conv_pv_edge
    else:
        best = 'reversal'
        best_terminal_profit = rev_terminal_profit
        best_pv_edge = rev_pv_edge

    return {
        'K': float(K), 'call': c, 'put': p, 'days': days, 'r': r,
        'growth': growth, 'pv_k': pv_k,
        'conv_cost': conv_cost, 'conv_pv_edge': conv_pv_edge,
        'conv_terminal_profit': conv_terminal_profit,
        'rev_proceeds': rev_proceeds, 'rev_pv_edge': rev_pv_edge,
        'rev_terminal_profit': rev_terminal_profit,
        'best': best, 'best_terminal_profit': best_terminal_profit,
        'best_pv_edge': best_pv_edge,
    }

a210 = arb_stats(ATM)

print(f"Spot S = ${S:,.2f}")
print(f"Strike K = ${ATM:,.2f}")
print(f"Contracts = {N_CONTRACTS:,}  ->  shares = {N_SHARES:,}")
print(f"Days = {a210['days']:.0f}, finance rate = {a210['r']:.4%}")
print()
print("Executable option quotes:")
print(f"  Call: bid ${float(a210['call']['bid']):.2f}, ask ${float(a210['call']['ask']):.2f}")
print(f"  Put : bid ${float(a210['put']['bid']):.2f}, ask ${float(a210['put']['ask']):.2f}")
print(f"  PV(K) = ${a210['pv_k']:.4f} per share")
print()

if a210['best'] == 'conversion':
    print("Best trade: CONVERSION")
    print(f"  Long  {N_CONTRACTS:,} puts at the ask")
    print(f"  Short {N_CONTRACTS:,} calls at the bid")
    print(f"  Long  {N_SHARES:,} shares at spot")
    print(f"  Borrow the net package cost: ${a210['conv_cost'] * N_SHARES:,.2f}")
else:
    print("Best trade: REVERSAL")
    print(f"  Long  {N_CONTRACTS:,} calls at the ask")
    print(f"  Short {N_CONTRACTS:,} puts at the bid")
    print(f"  Short {N_SHARES:,} shares at spot")
    print(f"  Invest the net package proceeds: ${a210['rev_proceeds'] * N_SHARES:,.2f}")


### 1.3.

What is the expected PnL of your trade? Detail what you expect to make in each leg of the trade.

**Answer 1.3.** The option/stock package has a terminal value that is independent of the terminal stock price: \(+K\) per share for a conversion and \(-K\) per share for a reversal. Financing the package therefore leaves a deterministic terminal P&L if all trades can actually be executed at the displayed quotes.


In [ ]:
if a210['best'] == 'conversion':
    rows = [
        ['Long stock', -S * N_SHARES, r'$+S_T \times N$'],
        ['Long put', -float(a210['put']['ask']) * N_SHARES,
         r'$+\max(K-S_T,0)\times N$'],
        ['Short call', +float(a210['call']['bid']) * N_SHARES,
         r'$-\max(S_T-K,0)\times N$'],
        ['Borrow package cost', +a210['conv_cost'] * N_SHARES,
         f"-${a210['conv_cost'] * a210['growth'] * N_SHARES:,.2f}"],
    ]
    terminal_package = ATM * N_SHARES
    terminal_financing = -a210['conv_cost'] * a210['growth'] * N_SHARES
    pv_profit = a210['conv_pv_edge'] * N_SHARES
else:
    rows = [
        ['Short stock', +S * N_SHARES, r'$-S_T \times N$'],
        ['Long call', -float(a210['call']['ask']) * N_SHARES,
         r'$+\max(S_T-K,0)\times N$'],
        ['Short put', +float(a210['put']['bid']) * N_SHARES,
         r'$-\max(K-S_T,0)\times N$'],
        ['Invest package proceeds', -a210['rev_proceeds'] * N_SHARES,
         f"+${a210['rev_proceeds'] * a210['growth'] * N_SHARES:,.2f}"],
    ]
    terminal_package = -ATM * N_SHARES
    terminal_financing = a210['rev_proceeds'] * a210['growth'] * N_SHARES
    pv_profit = a210['rev_pv_edge'] * N_SHARES

leg_table = pd.DataFrame(rows, columns=['Leg', 'Initial cash flow ($)', 'Expiration payoff'])
display(leg_table)

print(f"Net initial cash flow after financing: $0.00")
print(f"Stock + option package payoff at expiration: ${terminal_package:,.2f}")
print(f"Financing/cash payoff at expiration:         ${terminal_financing:,.2f}")
print(f"Guaranteed terminal P&L:                     ${a210['best_terminal_profit']:,.2f}")
print(f"Present-value edge:                           ${pv_profit:,.2f}")


### 1.4.

Is this an arbitrage? What risks are there?

Specify whether you are trading a **conversion** (short call, long put) or **reversal** (reverse conversion - long call, short put.)

**Answer 1.4.** The cell below gives the classification mechanically. A positive terminal P&L after crossing the bid/ask spread is an arbitrage **in the simplified model**; a non-positive P&L is not.

In practice, even a displayed positive edge is not automatically a realizable arbitrage. The main risks are simultaneous-execution/slippage risk, transaction fees, insufficient displayed size for a 1,000-contract order, stock-borrow availability and borrow cost for a reversal, early exercise/assignment of American options, dividend/carry mismatch, and stale or non-firm quotes. These matter because put-call parity is exact only under the financing/dividend/exercise assumptions being used.


In [ ]:
if a210['best_terminal_profit'] > 0:
    print(
        f"At the displayed quotes the {a210['best'].upper()} has a "
        f"positive modeled terminal P&L of ${a210['best_terminal_profit']:,.2f}."
    )
    print("So it is a model arbitrage if every leg can be executed as assumed.")
else:
    print(
        f"Neither executable direction produces a positive modeled P&L. "
        f"The better package is the {a210['best'].upper()}, with "
        f"${a210['best_terminal_profit']:,.2f} terminal P&L."
    )
    print("Therefore the ATM quote is not an executable arbitrage at these bid/ask prices.")


### 1.5.

Suppose that the `ATM strike = 210` trade is not an arbitrage, but rather an indication about the true financing rate.

Instead of using the given financing rate, solve for a financing rate which sets the profits (at this strike) to 0, keeping put-call parity.

Recall that we're calculating a financing rate with ACT/360 annual compounding.

**Answer 1.5.** Set the chosen package's profit equal to zero. If \(A\) is the package's implied present value of the strike, then
\[
A=\frac{K}{1+\frac{\text{days}}{360}r}
\quad\Longrightarrow\quad
r=\frac{360}{\text{days}}\left(\frac{K}{A}-1\right).
\]
For a conversion, \(A=S+P_{\rm ask}-C_{\rm bid}\). For a reversal, \(A=S-C_{\rm ask}+P_{\rm bid}\).


In [ ]:
if a210['best'] == 'conversion':
    implied_pv_k = a210['conv_cost']
else:
    implied_pv_k = a210['rev_proceeds']

implied_r_210 = (360 / a210['days']) * (ATM / implied_pv_k - 1)

print(f"Package used: {a210['best'].upper()}")
print(f"Implied PV(K): ${implied_pv_k:.4f}")
print(f"Zero-profit ACT/360 financing rate: {implied_r_210:.6%}")
print(f"Given financing rate:                 {a210['r']:.6%}")
print(f"Difference:                           {(implied_r_210-a210['r'])*1e4:.2f} bp")


### 1.6.

Consider the strike at `270`. What is your expected profit from that arbitrage?

Aside from the financing rate, point to specific data which concerns you about putting on this trade.

**Answer 1.6.** The same executable conversion/reversal comparison is applied at \(K=270\). The table then surfaces the market-data fields that matter for whether the apparent edge can actually be traded.


In [ ]:
K270 = 270.0
a270 = arb_stats(K270)

print(f"Best modeled trade at K=270: {a270['best'].upper()}")
print(f"Guaranteed terminal P&L for 1,000 contracts: ${a270['best_terminal_profit']:,.2f}")
print(f"Present-value edge: ${a270['best_pv_edge'] * N_SHARES:,.2f}")
print()

concern_cols = ['last update date', 'last update time', 'bid', 'ask',
                'bid size', 'ask size', 'open int', 'volume']
q270 = pd.DataFrame(
    [a270['call'][concern_cols], a270['put'][concern_cols]],
    index=['Call 270', 'Put 270']
).copy()
q270['spread'] = q270['ask'].astype(float) - q270['bid'].astype(float)
q270['spread / mid'] = q270['spread'] / (
    (q270['ask'].astype(float) + q270['bid'].astype(float)) / 2
)
display(q270)

print("Specific concerns:")
for label, row in q270.iterrows():
    print(
        f"  {label}: displayed sizes are bid {float(row['bid size']):,.0f} / "
        f"ask {float(row['ask size']):,.0f} contracts versus a 1,000-contract target; "
        f"volume={float(row['volume']):,.0f}, open interest={float(row['open int']):,.0f}; "
        f"spread=${float(row['spread']):.2f} ({float(row['spread / mid']):.2%} of mid); "
        f"last update={row['last update date']} {row['last update time']}."
    )

print(
    "\nThe apparent arbitrage is only compelling if enough size is actually executable. "
    "Wide spreads, small displayed size, low volume/open interest, or a stale timestamp "
    "can make the quoted 1,000-contract profit unattainable."
)


# 2. Option Value

### 2.1.

For the `2026-09-18` expiration, create a plot of...
* Moneyness Ratio, $S/K$, on the x-axis
* call value on the y-axis
* put value on the y-axis

How would you describe the relationship? Does it look like the plots of the options' final payoffs?

**Answer 2.1.** Current option values do **not** look exactly like their expiration payoff diagrams. Calls become more valuable as \(S/K\) rises and puts become less valuable, but before expiration both contain time value, so the curves are smooth rather than the piecewise-linear hockey sticks of final payoff.


In [ ]:
values = pd.DataFrame(index=strikes)
values.index.name = 'strike'
values['moneyness S/K'] = S / values.index.to_numpy(dtype=float)
values['call value'] = calls.loc[strikes, 'price'].astype(float)
values['put value'] = puts.loc[strikes, 'price'].astype(float)
values = values.sort_values('moneyness S/K')

plt.figure()
plt.plot(values['moneyness S/K'], values['call value'], marker='o', label='Call')
plt.plot(values['moneyness S/K'], values['put value'], marker='o', label='Put')
plt.axvline(1, linewidth=1)
plt.xlabel('Moneyness ratio S/K')
plt.ylabel('Option value ($/share)')
plt.title(f'{TICK} {EXPRY}: Option Value vs. Moneyness')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


### 2.2.

Let's now look across multiple expirations, at a single strike.

For the `ATM strike=210`, make a plot of...
* expiration date (or days to expiration) on the x-axis
* call value and put values on the y-axis

What do you notice in this relationship?

**Answer 2.2.** Holding the strike fixed at 210 isolates the effect of time to expiration. The plot below reads every date-named sheet in the workbook and extracts the 210 call and put. In general, longer-dated options tend to have more time value, although the relationship need not be perfectly monotone because volatility, rates, dividends/carry, and the underlying price can differ across maturities.


In [ ]:
xls = pd.ExcelFile(FILEDATA)
term_rows = []

for sheet in xls.sheet_names:
    expiry = pd.to_datetime(sheet, errors='coerce')
    if pd.isna(expiry):
        continue

    data = pd.read_excel(FILEDATA, sheet_name=sheet)
    data.rename(columns={'Unnamed: 0': 'ticker'}, inplace=True)

    k_mask = np.isclose(data['strike price'].astype(float), ATM)
    c_mask = data['option type'].astype(str).str.lower().eq('call')
    p_mask = data['option type'].astype(str).str.lower().eq('put')

    crows = data.loc[k_mask & c_mask]
    prows = data.loc[k_mask & p_mask]
    if len(crows) != 1 or len(prows) != 1:
        continue

    c = crows.iloc[0]
    p = prows.iloc[0]
    term_rows.append({
        'expiration': expiry,
        'days to expiration': float(np.mean([
            c['days to expiration'], p['days to expiration']
        ])),
        'call value': float(c['price']),
        'put value': float(p['price']),
    })

term = pd.DataFrame(term_rows).sort_values('expiration')
display(term)

plt.figure()
plt.plot(term['expiration'], term['call value'], marker='o', label='Call K=210')
plt.plot(term['expiration'], term['put value'], marker='o', label='Put K=210')
plt.xlabel('Expiration')
plt.ylabel('Option value ($/share)')
plt.title(f'{TICK}: Value Across Expirations at K=210')
plt.legend()
plt.grid(alpha=0.25)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


# 3. Building an Options Position

Use the `2026-09-18` market data.

### 3.1. 

For each of the following trades, calculate the cost of putting it on. 

### 3.2.

In a sentence, describe what exposure the position has. (How would it profit or lose?)

### 3.3. 

Plot the payoff of each of the three strategies.

#### Contract Size
Remember that a contract of calls or puts is for 100 units (shares)

| Package | Component | Position | Quantity | Strike |
|---------|-----------|----------|----------|--------|
| **<span style="color: midnightblue;">A</span>** | <span style="color: midnightblue;">Shares</span> | — | 0 | — |
|       | <span style="color: midnightblue;">Call</span> | **<span style="color: green;">Long</span>** | 1 | 210 |
|       | <span style="color: midnightblue;">Put</span> | **<span style="color: green;">Long</span>** | 1 | 210 |
|         |           |          |          |        |
| **<span style="color: steelblue;">B</span>** | <span style="color: steelblue;">Shares</span> | — | 0 | — |
|       | <span style="color: steelblue;">Call</span> | **<span style="color: red;">Short</span>** | 1 | 245 |
|       | <span style="color: steelblue;">Put</span> | **<span style="color: green;">Long</span>** | 1 | 195 |
|         |           |          |          |        |
| **<span style="color: cadetblue;">C</span>** | <span style="color: cadetblue;">Shares</span> | **<span style="color: green;">Long</span>** | 100 | — |
|       | <span style="color: cadetblue;">Call</span> | **<span style="color: red;">Short</span>** | 1 | 245 |
|       | <span style="color: cadetblue;">Put</span> | **<span style="color: green;">Long</span>** | 1 | 195 |

**Answers 3.1–3.3.** I use executable prices: ask for a long option, bid for a short option, and the supplied spot price for shares. One option contract controls 100 shares.

- **Package A:** long 210 call + long 210 put = a **long straddle**. It benefits from a sufficiently large move in either direction; its maximum loss is the premium if the stock finishes near 210.
- **Package B:** short 245 call + long 195 put. It is **bearish/asymmetric**: it gains terminal option value on a large downside move, is flat in option payoff between 195 and 245, and loses without bound above 245 because of the naked short call.
- **Package C:** long 100 shares + long 195 put − short 245 call = a **collar**. It participates in stock gains between the strikes, has a downside floor from the put, and gives up upside above 245 to the short call.

The first table gives the cash cost to initiate each package. The charts show expiration P&L after subtracting that initial cost (financing ignored), which makes the profit/loss exposure directly visible.


In [ ]:
def bid(K, kind):
    return float(option_row(K, kind)['bid'])

def ask(K, kind):
    return float(option_row(K, kind)['ask'])

# Initial cash cost; positive = cash paid, negative = cash received.
cost_A = CONTRACT_SIZE * (ask(210, 'call') + ask(210, 'put'))
cost_B = CONTRACT_SIZE * (-bid(245, 'call') + ask(195, 'put'))
cost_C = CONTRACT_SIZE * S + cost_B

package_costs = pd.DataFrame({
    'Package': ['A', 'B', 'C'],
    'Initial cost ($)': [cost_A, cost_B, cost_C],
}).set_index('Package')

display(package_costs.style.format('${:,.2f}'))

ST = np.linspace(max(0, S * 0.45), S * 1.60, 600)

payoff_A = CONTRACT_SIZE * (
    np.maximum(ST - 210, 0) + np.maximum(210 - ST, 0)
)
payoff_B = CONTRACT_SIZE * (
    -np.maximum(ST - 245, 0) + np.maximum(195 - ST, 0)
)
payoff_C = CONTRACT_SIZE * ST + payoff_B

pnl_A = payoff_A - cost_A
pnl_B = payoff_B - cost_B
pnl_C = payoff_C - cost_C

packages = {
    'A — Long 210 Straddle': pnl_A,
    'B — Short 245 Call + Long 195 Put': pnl_B,
    'C — 195/245 Collar on 100 Shares': pnl_C,
}

for title, pnl in packages.items():
    plt.figure()
    plt.plot(ST, pnl)
    plt.axhline(0, linewidth=1)
    plt.axvline(S, linestyle='--', linewidth=1, label=f'Current spot = {S:.2f}')
    plt.xlabel('Stock price at expiration')
    plt.ylabel('Expiration P&L ($)')
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()
